<a href="https://colab.research.google.com/github/Vivek-Chaudhari30/GRPO-Zero/blob/main/notebooks/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GRPO-Zero on Colab — train Qwen2.5 on GSM8K with RLVR

From-scratch **GRPO** (Group Relative Policy Optimization) + a verifiable reward (RLVR) on GSM8K.
Repo: https://github.com/Vivek-Chaudhari30/GRPO-Zero

**Before running:** `Runtime > Change runtime type > GPU` (T4 is enough for 0.5B; L4/A100 for 1.5B).

Pipeline: verify env → unit tests → overfit sanity (M4) → baseline eval → full GRPO train (M5) → eval trained policy + curves (M6).

In [1]:
!nvidia-smi

Wed Jun 17 11:28:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone the repo and install deps (Colab already ships a CUDA torch — don't reinstall it).
!git clone https://github.com/Vivek-Chaudhari30/GRPO-Zero.git
%cd GRPO-Zero
!pip install -q transformers datasets peft accelerate trl pyyaml matplotlib

Cloning into 'GRPO-Zero'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 66 (delta 23), reused 62 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 65.42 KiB | 956.00 KiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/GRPO-Zero
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.4 MB/s eta 0:00:00


In [3]:
# Sanity: the verifier + GRPO-math unit tests must pass on the box too.
!python -m pytest -q tests/

.....................................................                    [100%]
53 passed in 4.85s


In [14]:
pip uninstall -y torchao


## Smoke test — does the GPU box run a full GRPO step?
Tiny and fast (~1 min on a T4): 2 prompts, G=4, 3 steps, short completions. Confirms generation + the update + adapter save all work on this machine before the longer runs. Expect it to just **finish without error** — rewards may be ~0 with such short completions, which is fine for a smoke test.

In [15]:
!python -m src.train --overfit 2 --steps 3 --group-size 4 --prompts-per-step 2 \
    --max-new-tokens 128 --lr 5e-5 --save-dir /tmp/smoke --log /tmp/smoke_log.json

Device: cuda | model: Qwen/Qwen2.5-0.5B-Instruct
G=4 prompts/step=2 max_new_tokens=128 lr=5e-05 kl_coef=0.04 steps=3 | OVERFIT n=2
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 6221.44it/s]
LoRA trainable params: 8,798,208 / 502,830,976 (1.75%)
README.md: 100% 7.93k/7.93k [00:00<00:00, 27.9MB/s]
main/train-00000-of-00001.parquet: 100% 2.31M/2.31M [00:00<00:00, 3.01MB/s]
main/test-00000-of-00001.parquet: 100% 419k/419k [00:00<00:00, 2.03MB/s]
Generating train split: 100% 7473/7473 [00:00<00:00, 341100.15 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 283300.23 examples/s]
Prompt pool: 2 problems

step   0 | reward 0.138 (correct 0.12) | loss -0.0000 | KL 0.0000 | clip 0.00 | |g| 0.71 | len 128 | 15s
step   1 | reward 0.013 (correct 0.00) | loss -0.0002 | KL 0.0012 | clip 0.01 | |g| 1.19 | len 122 | 29s
step   2 | reward 0.025 (correct 0.00) | loss +0.0014 | KL 0.0078 | clip 0.00 | |g| 1.08 | len 126 | 44s

D

## M4 — Overfit sanity check
Fix 16 prompts and train only on them. The loop is working if **mean reward climbs** and **KL grows from ~0** (it starts at 0 because the LoRA adapter initializes to zero, so the policy begins identical to the reference).

In [16]:
# expandable_segments reduces CUDA fragmentation OOMs on the T4.
# Lighter than the full run: prompts/step=4 and 400 tokens to keep T4 steps reasonable.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python -m src.train --overfit 16 --steps 30 --group-size 8 --prompts-per-step 4 \
    --max-new-tokens 400 --micro-batch 2 --lr 5e-5 \
    --save-dir checkpoints/overfit --log results/overfit_log.json
!python scripts/plot_curves.py results/overfit_log.json results/overfit_curve.png

Device: cuda | model: Qwen/Qwen2.5-0.5B-Instruct
G=8 prompts/step=4 max_new_tokens=400 lr=5e-05 kl_coef=0.04 steps=30 | OVERFIT n=16
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 5531.04it/s]
LoRA trainable params: 8,798,208 / 502,830,976 (1.75%)
Prompt pool: 16 problems

step   0 | reward 0.347 (correct 0.28) | loss +0.0000 | KL 0.0000 | clip 0.00 | |g| 0.67 | len 248 | 84s
step   1 | reward 0.322 (correct 0.25) | loss +0.0006 | KL 0.0010 | clip 0.01 | |g| 0.44 | len 283 | 183s
step   2 | reward 0.062 (correct 0.03) | loss -0.0009 | KL 0.0019 | clip 0.01 | |g| 0.87 | len 318 | 284s
step   3 | reward 0.491 (correct 0.44) | loss +0.0007 | KL 0.0042 | clip 0.01 | |g| 0.62 | len 251 | 377s
step   4 | reward 0.547 (correct 0.47) | loss +0.0012 | KL 0.0096 | clip 0.01 | |g| 0.63 | len 204 | 469s
step   5 | reward 0.416 (correct 0.34) | loss +0.0003 | KL 0.0108 | clip 0.01 | |g| 0.47 | len 255 | 569s
step   6 | reward 0.044 (corr

## Baseline — base model pass@1 on the full GSM8K test set
This is the number the trained policy must beat. (~1319 problems; tens of minutes on T4.)

In [17]:
!python -m src.eval --config configs/grpo_gsm8k.yaml --n 1319 --tag baseline_full \
    --save-completions results/baseline_completions_full.json

Loading Qwen/Qwen2.5-0.5B-Instruct ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 24634.40it/s]
Device: cuda, dtype: torch.bfloat16
Evaluating pass@1 on 1319 GSM8K test problems (max_new_tokens=512)
  [8/1319] running pass@1 acc=0.500  (27s)
  [16/1319] running pass@1 acc=0.375  (51s)
  [24/1319] running pass@1 acc=0.375  (70s)
  [32/1319] running pass@1 acc=0.406  (90s)
  [40/1319] running pass@1 acc=0.450  (117s)
  [48/1319] running pass@1 acc=0.438  (145s)
  [56/1319] running pass@1 acc=0.464  (167s)
  [64/1319] running pass@1 acc=0.453  (192s)
  [72/1319] running pass@1 acc=0.458  (211s)
  [80/1319] running pass@1 acc=0.463  (234s)
  [88/1319] running pass@1 acc=0.489  (254s)
  [96/1319] running pass@1 acc=0.479  (282s)
  [104/1319] running pass@1 acc=0.471  (303s)
  [112/1319] running pass@1 acc=0.473  (331s)
  [120/1319] running pass@1 acc=0.467  (354s)
  [128/1319] running pass@1 acc=0.469  (374s)
  [136/1319] run

## M5 — Full GRPO training run
Trains the LoRA policy on the GSM8K train split. Tune `--steps` / `--group-size` to your GPU and time budget (T4 is slower than L4/A100). Checkpoints land in `checkpoints/grpo_lora`.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python -m src.train --config configs/grpo_gsm8k.yaml \
    --save-dir checkpoints/grpo_lora --log results/train_log.json
!python scripts/plot_curves.py results/train_log.json results/reward_curve.png

Device: cuda | model: Qwen/Qwen2.5-0.5B-Instruct
G=8 prompts/step=8 max_new_tokens=512 lr=1e-06 kl_coef=0.04 steps=200
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 5563.63it/s]
LoRA trainable params: 8,798,208 / 502,830,976 (1.75%)
Prompt pool: 1600 problems

step   0 | reward 0.319 (correct 0.25) | loss +0.0000 | KL 0.0000 | clip 0.00 | |g| 0.59 | len 298 | 227s
step   1 | reward 0.280 (correct 0.22) | loss +0.0078 | KL 0.0010 | clip 0.01 | |g| 6.11 | len 319 | 456s
step   2 | reward 0.195 (correct 0.12) | loss +0.0003 | KL 0.0010 | clip 0.01 | |g| 0.38 | len 355 | 676s
step   3 | reward 0.339 (correct 0.28) | loss +0.0002 | KL 0.0009 | clip 0.01 | |g| 0.30 | len 336 | 898s
step   4 | reward 0.459 (correct 0.39) | loss +0.0000 | KL 0.0009 | clip 0.01 | |g| 0.33 | len 300 | 1129s
step   5 | reward 0.286 (correct 0.22) | loss -0.0011 | KL 0.0009 | clip 0.01 | |g| 1.59 | len 307 | 1349s
step   6 | reward 0.398 (correct 0.33)

## M6 — Eval the trained policy + before/after
Same eval, now with the trained LoRA adapter loaded.

In [ ]:
!python -m src.eval --config configs/grpo_gsm8k.yaml --n 1319 --tag trained \
    --adapter checkpoints/grpo_lora --save-completions results/trained_completions.json

In [ ]:
import json
from IPython.display import Image, display

m = json.load(open('results/metrics.json'))
b = m.get('baseline_full', m.get('baseline'))
t = m.get('trained')
print(f"baseline pass@1 = {b['accuracy']}  (n={b['n']})")
if t:
    print(f"trained  pass@1 = {t['accuracy']}  (n={t['n']})")
    print(f"delta           = {t['accuracy'] - b['accuracy']:+.4f}")
display(Image('results/reward_curve.png'))

## Save results back (optional)
Commit the real numbers + curves to the repo so the README can cite them.
```python
# from google.colab import files; files.download('results/reward_curve.png')
# or: configure git creds and push results/ + checkpoints metadata
```